In [ ]:
# ============================================================
# 5. Enhancing K-Means Clustering with Association Rule Mining
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.optimize import linear_sum_assignment

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [ ]:
# ============================================================
# Load dataset
# ============================================================

df = pd.read_csv("mobile_price.csv")

X = df.drop("price_range", axis=1)
y = df["price_range"]

feature_cols = X.columns.tolist()
label_col = "price_range"

print("Feature shape:", X.shape)
print("Label shape:", y.shape)
print("Number of classes:", y.nunique())

In [ ]:
# ============================================================
# Helper: map cluster IDs to true labels using Hungarian Algorithm
# ============================================================

def map_clusters_to_labels(y_true, cluster_labels):
    y_true = np.array(y_true)
    cluster_labels = np.array(cluster_labels)

    true_classes = np.unique(y_true)
    clusters = np.unique(cluster_labels)

    cost_matrix = np.zeros((len(clusters), len(true_classes)), dtype=int)

    for i, cluster in enumerate(clusters):
        for j, true_class in enumerate(true_classes):
            cost_matrix[i, j] = np.sum(
                (cluster_labels == cluster) & (y_true == true_class)
            )

    row_ind, col_ind = linear_sum_assignment(-cost_matrix)

    cluster_to_label = {}
    for row, col in zip(row_ind, col_ind):
        cluster_to_label[clusters[row]] = true_classes[col]

    mapped_labels = np.array([cluster_to_label[c] for c in cluster_labels])

    return mapped_labels, cluster_to_label

In [ ]:
# ============================================================
# Helper: evaluate clustering as classification after mapping
# ============================================================

def evaluate_clustering(y_true, cluster_labels):
    mapped_labels, mapping = map_clusters_to_labels(y_true, cluster_labels)

    acc = accuracy_score(y_true, mapped_labels)
    precision = precision_score(y_true, mapped_labels, average="macro", zero_division=0)
    recall = recall_score(y_true, mapped_labels, average="macro", zero_division=0)
    f1 = f1_score(y_true, mapped_labels, average="macro", zero_division=0)

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Cluster Mapping": mapping
    }

In [ ]:
# ============================================================
# Original K-Means using standardized original features
# ============================================================

seeds = [0, 10, 42, 100, 999]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

baseline_results = []

for seed in seeds:
    kmeans = KMeans(
        n_clusters=4,
        random_state=seed,
        n_init=10
    )

    clusters = kmeans.fit_predict(X_scaled)

    result = evaluate_clustering(y, clusters)
    result["Method"] = "Original K-Means"
    result["Seed"] = seed

    baseline_results.append(result)

baseline_df = pd.DataFrame(baseline_results)

baseline_metrics = baseline_df[
    ["Seed", "Method", "Accuracy", "Precision", "Recall", "F1-score"]
]

print("===== Original K-Means Results =====")
display(baseline_metrics)

In [ ]:
# ============================================================
# Discretize all features into low / medium / high
# using 3:4:3 ratio
# ============================================================

def categorize_343(series):
    min_val = series.min()
    max_val = series.max()
    value_range = max_val - min_val

    low_threshold = min_val + 0.3 * value_range
    high_threshold = min_val + 0.7 * value_range

    def categorize_value(x):
        if x <= low_threshold:
            return "low"
        elif x <= high_threshold:
            return "medium"
        else:
            return "high"

    return series.apply(categorize_value)


X_discrete = pd.DataFrame()

for col in feature_cols:
    X_discrete[col] = categorize_343(X[col])

print("Discretized features:")
display(X_discrete.head())

In [ ]:
# ============================================================
# Convert each sample into transaction format
# Include all discretized features and the label item
# ============================================================

transactions = []

for i in range(len(X_discrete)):
    transaction = []

    for col in feature_cols:
        transaction.append(f"{col}_{X_discrete.iloc[i][col]}")

    transaction.append(f"price_range_{y.iloc[i]}")

    transactions.append(transaction)

print("Example transaction:")
print(transactions[0])

In [ ]:
# ============================================================
# Apply FP-growth
# ============================================================

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

transaction_df = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = fpgrowth(
    transaction_df,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

print("Number of frequent itemsets:", len(frequent_itemsets))
display(frequent_itemsets.head(10))

In [ ]:
# ============================================================
# Generate association rules
# ============================================================

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.2
)

print("Number of association rules:", len(rules))
display(rules.head())

In [ ]:
# ============================================================
# Keep only rules whose consequent is exactly one price_range_x
# ============================================================

label_items = {f"price_range_{c}" for c in sorted(y.unique())}

def is_label_rule(row):
    antecedents = set(row["antecedents"])
    consequents = set(row["consequents"])

    # antecedent 不可以含 price_range
    no_label_in_antecedent = len(antecedents & label_items) == 0

    # consequent 必須剛好只有一個 item，且這個 item 是 price_range_x
    consequent_is_single_label = (
        len(consequents) == 1 and
        list(consequents)[0] in label_items
    )

    return no_label_in_antecedent and consequent_is_single_label


label_rules = rules[rules.apply(is_label_rule, axis=1)].copy()

label_rules = label_rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].copy()

label_rules = label_rules.sort_values(
    by=["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

print("Number of label-predicting rules:", len(label_rules))
display(label_rules.head(20))
print("Number of label-predicting rules:", len(label_rules))
display(label_rules.head(20))

In [ ]:
# ============================================================
# Convert label rules into ARM-based label score features
# ============================================================

def build_arm_score_features(transactions_without_label, label_rules, class_labels):
    score_matrix = np.zeros((len(transactions_without_label), len(class_labels)))

    class_to_index = {
        int(c): idx for idx, c in enumerate(class_labels)
    }

    parsed_rules = []

    for _, row in label_rules.iterrows():
        antecedent = set(row["antecedents"])
        consequent_set = set(row["consequents"])

        # 安全取出 price_range_x
        label_consequents = [
            item for item in consequent_set
            if item.startswith("price_range_")
        ]

        # 如果沒有 label consequent，跳過
        if len(label_consequents) != 1:
            continue

        consequent = label_consequents[0]
        target_class = int(consequent.replace("price_range_", ""))

        rule_score = row["confidence"] * row["lift"]

        parsed_rules.append({
            "antecedent": antecedent,
            "target_class": target_class,
            "rule_score": rule_score
        })

    for i, transaction in enumerate(transactions_without_label):
        transaction_set = set(transaction)

        for rule in parsed_rules:
            if rule["antecedent"].issubset(transaction_set):
                class_idx = class_to_index[rule["target_class"]]
                score_matrix[i, class_idx] += rule["rule_score"]

    return score_matrix


# 建立不含 label 的 transaction
transactions_without_label = []

for i in range(len(X_discrete)):
    transaction = []

    for col in feature_cols:
        transaction.append(f"{col}_{X_discrete.iloc[i][col]}")

    transactions_without_label.append(transaction)


class_labels = sorted(y.unique())

arm_scores = build_arm_score_features(
    transactions_without_label,
    label_rules,
    class_labels
)

arm_score_df = pd.DataFrame(
    arm_scores,
    columns=[f"ARM_score_price_range_{c}" for c in class_labels]
)

print("ARM score features:")
display(arm_score_df.head())

In [ ]:
# ============================================================
# Build enhanced features
# Original standardized features + weighted ARM score features
# ============================================================

arm_scaler = StandardScaler()
arm_scores_scaled = arm_scaler.fit_transform(arm_scores)

alpha = 2.0

X_enhanced = np.hstack([
    X_scaled,
    alpha * arm_scores_scaled
])

print("Original feature shape:", X_scaled.shape)
print("Enhanced feature shape:", X_enhanced.shape)

In [ ]:
# ============================================================
# Improved K-Means using ARM-enhanced features
# ============================================================

improved_results = []

for seed in seeds:
    kmeans = KMeans(
        n_clusters=4,
        random_state=seed,
        n_init=10
    )

    clusters = kmeans.fit_predict(X_enhanced)

    result = evaluate_clustering(y, clusters)
    result["Method"] = "ARM-guided K-Means"
    result["Seed"] = seed

    improved_results.append(result)

improved_df = pd.DataFrame(improved_results)

improved_metrics = improved_df[
    ["Seed", "Method", "Accuracy", "Precision", "Recall", "F1-score"]
]

print("===== ARM-guided K-Means Results =====")
display(improved_metrics)

In [ ]:
# ============================================================
# Compare results
# ============================================================

all_results = pd.concat(
    [baseline_metrics, improved_metrics],
    axis=0
).reset_index(drop=True)

print("===== All Results =====")
display(all_results)

In [ ]:
# ============================================================
# Average performance comparison
# ============================================================

avg_results = all_results.groupby("Method")[
    ["Accuracy", "Precision", "Recall", "F1-score"]
].mean().reset_index()

std_results = all_results.groupby("Method")[
    ["Accuracy", "Precision", "Recall", "F1-score"]
].std().reset_index()

print("===== Average Performance =====")
display(avg_results)

print("===== Standard Deviation =====")
display(std_results)

In [ ]:
# ============================================================
# Visualize average performance comparison
# ============================================================

metrics = ["Accuracy", "Precision", "Recall", "F1-score"]

x = np.arange(len(metrics))
width = 0.35

original_values = avg_results[avg_results["Method"] == "Original K-Means"][metrics].values[0]
improved_values = avg_results[avg_results["Method"] == "ARM-guided K-Means"][metrics].values[0]

plt.figure(figsize=(9, 6))

plt.bar(
    x - width / 2,
    original_values,
    width,
    label="Original K-Means"
)

plt.bar(
    x + width / 2,
    improved_values,
    width,
    label="ARM-guided K-Means"
)

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Average Performance Comparison")
plt.legend()
plt.ylim(0, 1)
plt.grid(axis="y")
plt.show()

In [ ]:
# ============================================================
# Visualize F1-score across random seeds
# ============================================================

plt.figure(figsize=(8, 5))

for method in all_results["Method"].unique():
    subset = all_results[all_results["Method"] == method]

    plt.plot(
        subset["Seed"],
        subset["F1-score"],
        marker="o",
        label=method
    )

plt.xlabel("Random Seed")
plt.ylabel("F1-score")
plt.title("F1-score Across Random Seeds")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# PCA visualization of improved clustering
# ============================================================

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_enhanced_pca = pca.fit_transform(X_enhanced)

kmeans_final = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

final_clusters = kmeans_final.fit_predict(X_enhanced)

plot_df = pd.DataFrame({
    "PC1": X_enhanced_pca[:, 0],
    "PC2": X_enhanced_pca[:, 1],
    "Cluster": final_clusters,
    "True Label": y
})

plt.figure(figsize=(8, 6))

for cluster in sorted(plot_df["Cluster"].unique()):
    subset = plot_df[plot_df["Cluster"] == cluster]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("ARM-guided K-Means Clustering Visualized by PCA")
plt.legend()
plt.grid(True)
plt.show()